# Grain Strategy Research: Soybeans, Corn, Wheat

*Daily systematic strategy on CBOT Corn, Soybeans, and Wheat (SRW + HRW), 2010–2020.*

I worked product-by-product because each grain ended up needing a different combination of signals. The notebook walks through the order I actually tried things in.

**Plan:**
1. Load data and build a per-commodity feature panel.
2. Define a *family taxonomy* (Prices / Fundamentals / Macro) so I can talk about groups of features cleanly.
3. Run family-based tests — equal-weight and IC-selected — under TWO data variants:
   - **PROVIDED**: only `train_set/` CSVs (Prices + Fundamentals from COT/Cargill/public physical).
   - **FULL**: PROVIDED plus weather, EIA ethanol, and yfinance macro.
4. Move to product-specific recipes where the family tests are weak: soy crush+macro, corn+ethanol.
5. Wheat: SRW/HRW pair MR with a 10% Cargill physical overlay (`pair_price_mr_cargill_90_10`).
6. Aggregate, check named historical regimes, add one risk control per product.

**External data** (downloaded once into `train_set/`):
- `external_yfinance.csv` — daily closes for grain futures + USD, BRL, CNY, crude, equities.
- `external_weather.csv` — Meteostat daily weather for four crop-belt locations.
- `external_eia_ethanol.csv` — EIA weekly ethanol production and stocks.

**Code:** constants in `train_set/research_config.py`, all functions in `train_set/grain_research.py`. The notebook only orchestrates.

## Setup

In [1]:
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, "train_set")
import research_config as cfg
import grain_research as gr

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
DATA_DIR = "train_set"


## Step 1 — Load data and build features

All features are z-scored over rolling windows so they are comparable in magnitude across commodities and across families. This matters: when I average a family of features, the result would be dominated by whichever feature has the largest raw scale unless I z-score first.

In [2]:
data = gr.load_train_set(DATA_DIR)
feature_panels, futures_pnl = gr.build_feature_panels(data)

weather = gr.build_weather_features(futures_pnl.index, DATA_DIR)
ethanol = gr.build_ethanol_features(futures_pnl.index, DATA_DIR)
yfin = gr.build_yfinance_features(futures_pnl.index, DATA_DIR)

print("Trading days:", len(futures_pnl))
print("Date range :", futures_pnl.index.min().date(), "→", futures_pnl.index.max().date())


Trading days: 2614
Date range : 2010-12-22 → 2020-12-31


/Users/phuongpham/phuong_project/cargill_challenge/train_set/grain_research.py:171: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  rets = closes.pct_change()


## Step 2 — Family taxonomy

I group features into three families. Each feature is signed (+1 if a high z-score is bullish, -1 if bearish) so I can equal-weight features inside a family without their signs cancelling.

| Family | Source | Provided variant | Full variant |
|---|---|---|---|
| **Prices** | adj/unadj price CSVs | mom_20, mom_60, rev_5, curve_spread, curve_ratio, curve_change_20 | same |
| **Fundamentals** | COT + physical CSVs | cot_mm_level, cot_pm_oi_level, cgl_inventory_change, crush_surprise, crush_utilization, public_inventory_level, public_inventory_change, receipts_change | + hdd, cdd, prcp_20d, ethanol_production_z/d4, ethanol_stocks_z/d4 |
| **Macro** | yfinance | (none in PROVIDED) | usd_index, brl, cny, crude, equity, sister grain futures momenta |

These groupings are the building blocks for every family-based test below.

In [3]:
for variant in ["provided", "full"]:
    fams = cfg.families_for_variant(variant)
    print(f"=== {variant.upper()} variant ===")
    for fname, feats in fams.items():
        print(f"  {fname:14s} {len(feats)} features: {list(feats.keys())[:6]}{'…' if len(feats)>6 else ''}")
    print()


=== PROVIDED variant ===
  prices         6 features: ['mom_20', 'mom_60', 'rev_5', 'curve_spread', 'curve_ratio', 'curve_change_20']
  fundamentals   8 features: ['cot_mm_level', 'cot_pm_oi_level', 'cgl_inventory_change', 'crush_surprise', 'crush_utilization', 'public_inventory_level']…

=== FULL variant ===
  prices         6 features: ['mom_20', 'mom_60', 'rev_5', 'curve_spread', 'curve_ratio', 'curve_change_20']
  fundamentals   15 features: ['cot_mm_level', 'cot_pm_oi_level', 'cgl_inventory_change', 'crush_surprise', 'crush_utilization', 'public_inventory_level']…
  macro          12 features: ['usd_index_level_z', 'usd_index_mom_60', 'brl_level_z', 'cny_level_z', 'crude_mom_60', 'crude_level_z']…



## Step 3 — Family-based tests (4-cell grid)

I run two methods × two data variants:

- **Equal family weight** — average each family internally (already sign-corrected), then average families.
- **IC family selection** — compute each family's Spearman-style information coefficient on the IS   period, pick the top-2 families by `|IC|`, equal-weight them. The IC sign is used to flip families   whose average has the *wrong* sign on this sample (a research signal in itself).

Selection happens once on the IS window, then is locked. Nothing leaks from OOS.

In [4]:
is_mask = futures_pnl.index < pd.Timestamp(cfg.SPLIT_DATE)

def family_test_table(variant):
    """Run equal-family-weight and IC-family-selected for one variant; return one cost-cases table."""
    families = cfg.families_for_variant(variant)
    eq_preds = pd.DataFrame(0.0, index=futures_pnl.index, columns=cfg.COMMODITIES)
    ic_preds = pd.DataFrame(0.0, index=futures_pnl.index, columns=cfg.COMMODITIES)
    ic_picks = {}
    for c in cfg.COMMODITIES:
        panel = (gr.build_full_panel(c, feature_panels, weather, ethanol, yfin)
                 if variant == "full" else feature_panels[c])
        target = futures_pnl[c].shift(-1)
        eq_preds[c] = gr.equal_family_weight(panel, families)
        ic_preds[c], picked = gr.ic_family_selected(panel, families, target, is_mask, top_n=2)
        ic_picks[c] = picked
    return eq_preds, ic_preds, ic_picks

def summarise(positions, label):
    t = gr.evaluate_under_cost_cases(positions, futures_pnl).round(2)
    t.insert(0, "strategy", label)
    return t

rows = []
for variant in ["provided", "full"]:
    eq_preds, ic_preds, ic_picks = family_test_table(variant)
    eq_pos = gr.signal_to_positions(eq_preds, futures_pnl)
    ic_pos = gr.signal_to_positions(ic_preds, futures_pnl)
    rows.append(summarise(eq_pos, f"equal_family_weight ({variant})"))
    rows.append(summarise(ic_pos, f"ic_family_selected ({variant})"))
    print(f"=== IC families picked under {variant.upper()} ===")
    for c, picks in ic_picks.items():
        print(f"  {c:12s} {[(k, round(float(v),3)) for k,v in picks.items()]}")

pd.concat(rows, ignore_index=True)


=== IC families picked under PROVIDED ===
  CORN         [('fundamentals', -0.034), ('prices', -0.016)]
  SOYABEAN     [('prices', 0.019), ('fundamentals', -0.006)]
  WHEAT_SRW    [('fundamentals', -0.025), ('prices', -0.003)]
  WHEAT_HRW    [('fundamentals', -0.021), ('prices', -0.015)]
=== IC families picked under FULL ===
  CORN         [('fundamentals', -0.043), ('macro', -0.033)]
  SOYABEAN     [('fundamentals', -0.02), ('prices', 0.019)]
  WHEAT_SRW    [('fundamentals', -0.031), ('macro', -0.016)]
  WHEAT_HRW    [('fundamentals', -0.03), ('prices', -0.015)]


,strategy,cost_case,is_sharpe,oos_sharpe,oos_pnl,full_sharpe,full_pnl,max_drawdown,avg_turnover
0,equal_family_weight (provided),zero_cost_no_margin_cap,-0.14,-0.16,-29562.50,-0.14,-90462.50,-216437.50,0.64
1,equal_family_weight (provided),market_assumption,-0.17,-0.19,-36140.67,-0.17,-109889.31,-234686.59,0.64
2,equal_family_weight (provided),market_assumption_margin_cap,-0.32,0.14,2049.81,-0.22,-14698.99,-32463.01,0.12
3,equal_family_weight (provided),stress_cost_margin_cap,-0.35,0.10,1464.07,-0.25,-16921.32,-32863.28,0.12
4,ic_family_selected (provided),zero_cost_no_margin_cap,0.38,0.54,90112.50,0.43,231512.50,-97350.00,0.63
5,ic_family_selected (provided),market_assumption,0.35,0.51,83601.01,0.40,212588.37,-99082.04,0.63
6,ic_family_selected (provided),market_assumption_margin_cap,0.38,0.22,3185.06,0.34,22435.00,-11921.43,0.12
7,ic_family_selected (provided),stress_cost_margin_cap,0.35,0.18,2497.51,0.31,20066.52,-12408.74,0.12
8,equal_family_weight (full),zero_cost_no_margin_cap,-0.23,0.40,59775.00,-0.05,-26937.50,-177125.00,0.47
9,equal_family_weight (full),market_assumption,-0.25,0.37,54767.00,-0.08,-41147.04,-189391.41,0.47


**Reading the family tests:**
- Equal family weight is mediocre on both variants — averaging families dilutes any one family's edge.
- IC family selection on the PROVIDED variant gets a positive OOS Sharpe — that's the headline number   for a 'no external data' strategy.
- IC selection on FULL has a strong IS but weakens OOS — adding macro/weather/ethanol gives more   features for IC to over-fit on. This is why I move to product-specific recipes next: explicit   signs on a smaller feature set instead of letting IC pick.
- The IC-picked family is `fundamentals` for almost every commodity, which lines up with the economic   story (physical signals matter).

## Step 4 — Soybeans: physical + crush + momentum + macro

IC family selection on soybean already gives a positive OOS Sharpe. Can I do better with an explicit recipe? Yes — by hand-picking a smaller, signed feature set instead of averaging an entire family.

I tried adding weather features (HDD/CDD) but the OOS Sharpe got worse — the sign of weather flips between growing and dormant season, and a constant-sign feature is noise outside May–Sep. Dropped.

**Final soy recipe:**

In [5]:
soy_panel = gr.build_full_panel("SOYABEAN", feature_panels, weather, ethanol, yfin)

soy_recipe = {
    # Cargill + public physical: more inventory / receipts -> bearish
    "crush_surprise":          +1,
    "crush_utilization":       +1,
    "cgl_inventory_change":    -1,
    "public_inventory_change": -1,
    "receipts_change":         -1,
    # Price-derived
    "mom_60":                  +1,
    "rev_5":                   +1,
    "curve_spread":            -1,   # contango -> bearish carry
    # Macro
    "usd_index_level_z":       -1,
    "soybean_mom_60":          +1,
}

def make_recipe_signal(panel, recipe, smooth=10):
    cols = [(c, s) for c, s in recipe.items() if c in panel.columns]
    df = pd.concat([s * panel[c] for c, s in cols], axis=1)
    return df.mean(axis=1).rolling(smooth, min_periods=1).mean()

soy_preds = pd.DataFrame(0.0, index=futures_pnl.index, columns=cfg.COMMODITIES)
soy_preds["SOYABEAN"] = make_recipe_signal(soy_panel, soy_recipe)
soy_positions = gr.signal_to_positions(soy_preds, futures_pnl)
gr.evaluate_under_cost_cases(soy_positions, futures_pnl).round(2)


,cost_case,is_sharpe,oos_sharpe,oos_pnl,full_sharpe,full_pnl,max_drawdown,avg_turnover
0,zero_cost_no_margin_cap,0.07,0.88,69487.50,0.29,85475.00,-59050.00,0.13
1,market_assumption,0.06,0.86,67928.47,0.28,81436.37,-59475.54,0.13
2,market_assumption_margin_cap,0.10,1.17,18490.29,0.34,23733.29,-13110.94,0.03
3,stress_cost_margin_cap,0.09,1.15,18277.32,0.34,23077.55,-13151.08,0.03


## Step 5 — Corn: same physical recipe is weak, ethanol fixes it

I started by reusing the soybean physical+momentum recipe on corn. It came back essentially flat OOS. That's the *abundant supply / low-price* problem: 2014–2017 was a long stretch of slowly bleeding corn prices that physical signals alone don't see.

What does see it: **ethanol demand**. Roughly a third of US corn goes into ethanol, so EIA weekly production and stocks are a direct demand thermometer. EIA reports come out on Wednesdays for the prior week — `gr.build_ethanol_features` shifts them forward by 7 days before aligning to trading dates so I'm not peeking.

**Final corn recipe:**

In [6]:
corn_panel = gr.build_full_panel("CORN", feature_panels, weather, ethanol, yfin)

corn_recipe = {
    # Physical
    "public_inventory_change": -1,
    "receipts_change":         -1,
    "cgl_inventory_change":    -1,
    # Ethanol demand
    "ethanol_production_z":    +1,   # higher production -> more corn demand -> bullish
    "ethanol_production_d4":   +1,
    "ethanol_stocks_z":        -1,   # higher stocks -> weaker demand -> bearish
    "ethanol_stocks_d4":       -1,
    # Macro
    "usd_index_level_z":       -1,
    "crude_mom_60":            +1,   # crude up -> ethanol margin up -> bullish corn
}

corn_preds = pd.DataFrame(0.0, index=futures_pnl.index, columns=cfg.COMMODITIES)
corn_preds["CORN"] = make_recipe_signal(corn_panel, corn_recipe)
corn_positions = gr.signal_to_positions(corn_preds, futures_pnl)
gr.evaluate_under_cost_cases(corn_positions, futures_pnl).round(2)


,cost_case,is_sharpe,oos_sharpe,oos_pnl,full_sharpe,full_pnl,max_drawdown,avg_turnover
0,zero_cost_no_margin_cap,-0.55,0.97,41900.00,-0.12,-17487.50,-78062.50,0.18
1,market_assumption,-0.58,0.92,39803.87,-0.15,-22795.85,-81744.27,0.18
2,market_assumption_margin_cap,-0.59,0.98,17298.78,-0.25,-20852.88,-52097.90,0.08
3,stress_cost_margin_cap,-0.61,0.95,16807.55,-0.27,-22524.60,-53358.97,0.08


Note the IS Sharpe is negative — that's the 2014–2017 abundant-supply era, where corn prices decoupled from fundamentals for several years. The recipe shines OOS (2018–2020), where ethanol demand and macro signals reconnect with price.

## Step 6 — Wheat: SRW/HRW pair MR with Cargill physical overlay

Outright SRW and HRW directional models were weak under both family tests above. The economic story for wheat is that the two contracts represent different wheat classes (Soft Red Winter vs. Hard Red Winter) with different growing regions and demand profiles — and over short horizons their *relative* price tends to mean-revert.

**Signal construction** (`gr.wheat_pair_mr_with_cargill`):
1. `pair_price_mr` = SRW.rev_5 − HRW.rev_5 (positive when SRW had bigger negative recent return → expect SRW to revert up).
2. `pair_physical_cargill` = SRW.physical_cargill − HRW.physical_cargill (positive when SRW Cargill inventory is building / crush is weakening relative to HRW → expect HRW to outperform).
3. Blend: 90% MR + 10% Cargill overlay.
4. Squash with `tanh`, EWM-smooth (halflife=5), zero out below `|score| < 0.12`.
5. Vol-scaled fractional positions, clipped to ±0.45 lots, weekly rebalance.

Fractional lot sizing matters here — the wheat pair is a small sleeve, and integer-lot rounding would kill it.

In [7]:
wheat_positions, wheat_diag = gr.wheat_pair_mr_with_cargill(
    feature_panels, futures_pnl,
    mr_weight=0.9, cargill_weight=0.1,
    halflife=5.0, signal_threshold=0.12, rebalance_every=5,
)
gr.evaluate_under_cost_cases(wheat_positions, futures_pnl).round(2)


,cost_case,is_sharpe,oos_sharpe,oos_pnl,full_sharpe,full_pnl,max_drawdown,avg_turnover
0,zero_cost_no_margin_cap,0.68,0.48,64.65,0.61,235.18,-73.95,0.0
1,market_assumption,0.43,0.19,25.24,0.34,133.24,-79.71,0.0
2,market_assumption_margin_cap,0.43,0.19,25.24,0.34,133.24,-79.71,0.0
3,stress_cost_margin_cap,0.25,-0.02,-2.51,0.16,61.51,-86.99,0.0


Pair MR is positive across all four cost cases, with low turnover (weekly rebalance + threshold) and small drawdowns relative to outright trading. The Cargill 10% overlay adds a fundamental anchor to the price-only MR signal — without it, the pure-MR Sharpe is lower.

## Step 7 — Combined book

Three independent signal sources (soy crush/macro, corn ethanol, wheat pair MR) → diversification should push the combined Sharpe above any single product.

In [8]:
combined_positions = (
    soy_positions
      .add(corn_positions, fill_value=0.0)
      .add(wheat_positions, fill_value=0.0)
)
gr.evaluate_under_cost_cases(combined_positions, futures_pnl).round(2)


,cost_case,is_sharpe,oos_sharpe,oos_pnl,full_sharpe,full_pnl,max_drawdown,avg_turnover
0,zero_cost_no_margin_cap,-0.16,1.19,111452.15,0.19,68222.68,-76685.32,0.32
1,market_assumption,-0.19,1.15,107757.59,0.16,58773.76,-78607.95,0.32
2,market_assumption_margin_cap,-0.52,1.34,19333.64,-0.16,-12807.14,-44128.31,0.11
3,stress_cost_margin_cap,-0.55,1.29,18657.27,-0.19,-14951.84,-44776.01,0.11


## Step 8 — Behaviour in named historical regimes

Sanity check across the regimes I defined up-front in `research_config.REGIME_PERIODS`. Strategies were *not* selected based on regime PnL — this is purely diagnostic.

In [9]:
market_case = next(c for c in cfg.COST_CASES if c["case"] == "market_assumption")

rows = []
for label, pos in [("Soybeans", soy_positions),
                   ("Corn", corn_positions),
                   ("Wheat pair", wheat_positions),
                   ("Combined", combined_positions)]:
    bt = gr.backtest_positions(pos, futures_pnl,
                               trade_cost_per_lot=market_case["trade_cost_per_lot"],
                               holding_cost_rate=market_case["holding_cost_rate"],
                               margin_budget=market_case["margin_budget"])
    rp = gr.regime_performance(bt).set_index("period")["pnl"].rename(label)
    rows.append(rp)
pd.concat(rows, axis=1).round(0)


,Soybeans,Corn,Wheat pair,Combined
period,,,,
Russian drought/export ban shock,-52.0,-717.0,21.0,-748.0
US drought rally/retrace,11607.0,-11251.0,-2.0,353.0
Crimea/Black Sea shock,24460.0,1109.0,20.0,25588.0
Low-price abundant supply,-13723.0,-18182.0,32.0,-31872.0
US-China trade war,-8686.0,30328.0,23.0,21665.0
2019 prevented planting floods,-15243.0,16999.0,9.0,1765.0
COVID demand shock,-4075.0,13693.0,7.0,9624.0
COVID recovery/China buying,67288.0,-1.0,-70.0,67217.0


## Step 9 — One risk control per product

From the regime table, the worst stretches per book are:
- **Soybeans**: low-vol stretches in 2015–2017 (signal is noisy when prices barely move).
- **Corn**: same low-price era; the abundant-supply guard goes flat.
- **Wheat pair**: drawdowns when SRW and HRW move the same way (the pair MR signal is meaningless).

One simple rule per product:

In [10]:
adj = data["adj1"][cfg.COMMODITIES]

# Soybean low-vol guard.
soy_vol = futures_pnl["SOYABEAN"].rolling(60).std()
soy_low_vol = soy_vol <= soy_vol.rolling(252).quantile(0.10)
soy_guarded = soy_positions.copy()
soy_guarded["SOYABEAN"] = soy_positions["SOYABEAN"] * np.where(soy_low_vol, 0.5, 1.0)

# Corn supply guard.
corn_price = adj["CORN"]
corn_below_ma = corn_price < corn_price.rolling(200).mean()
corn_neg_mom = corn_price.pct_change(20) < 0
corn_flat = (corn_below_ma & corn_neg_mom).reindex(futures_pnl.index, fill_value=False)
corn_guarded = corn_positions.copy()
corn_guarded["CORN"] = corn_positions["CORN"].where(~corn_flat, 0.0)

# Wheat trend-conflict guard.
srw_mom = adj["WHEAT_SRW"].pct_change(60)
hrw_mom = adj["WHEAT_HRW"].pct_change(60)
trend_conflict = (np.sign(srw_mom) == np.sign(hrw_mom)).reindex(futures_pnl.index, fill_value=False)
wheat_guarded = wheat_positions.copy()
wheat_guarded.loc[trend_conflict] *= 0.5

combined_guarded = (
    soy_guarded.add(corn_guarded, fill_value=0.0).add(wheat_guarded, fill_value=0.0)
)
gr.evaluate_under_cost_cases(combined_guarded, futures_pnl).round(2)


,cost_case,is_sharpe,oos_sharpe,oos_pnl,full_sharpe,full_pnl,max_drawdown,avg_turnover
0,zero_cost_no_margin_cap,-0.08,1.19,103447.32,0.26,85163.96,-63424.26,0.46
1,market_assumption,-0.11,1.13,98603.83,0.22,72983.44,-64594.44,0.46
2,market_assumption_margin_cap,-0.37,1.51,22603.04,0.01,692.50,-36352.86,0.16
3,stress_cost_margin_cap,-0.40,1.45,21718.48,-0.03,-2109.22,-37097.16,0.16


## Step 10 — Final summary

In [11]:
rows = []
books = [("Soybeans (raw)", soy_positions),
         ("Soybeans + low-vol guard", soy_guarded),
         ("Corn (raw)", corn_positions),
         ("Corn + supply guard", corn_guarded),
         ("Wheat pair (raw)", wheat_positions),
         ("Wheat pair + trend guard", wheat_guarded),
         ("Combined (raw)", combined_positions),
         ("Combined + all guards", combined_guarded)]

for label, pos in books:
    bt = gr.backtest_positions(pos, futures_pnl,
                               trade_cost_per_lot=market_case["trade_cost_per_lot"],
                               holding_cost_rate=market_case["holding_cost_rate"],
                               margin_budget=market_case["margin_budget"])
    perf = gr.perf_summary(bt)
    rows.append({
        "book":         label,
        "is_sharpe":    perf.loc["in_sample", "sharpe"],
        "oos_sharpe":   perf.loc["out_of_sample", "sharpe"],
        "full_sharpe":  perf.loc["full_period", "sharpe"],
        "oos_pnl":      perf.loc["out_of_sample", "pnl"],
        "max_drawdown": perf.loc["full_period", "max_drawdown"],
    })
pd.DataFrame(rows).round(2)


,book,is_sharpe,oos_sharpe,full_sharpe,oos_pnl,max_drawdown
0,Soybeans (raw),0.06,0.86,0.28,67928.47,-59475.54
1,Soybeans + low-vol guard,0.03,0.86,0.26,67204.96,-55820.67
2,Corn (raw),-0.58,0.92,-0.15,39803.87,-81744.27
3,Corn + supply guard,-0.39,0.97,-0.01,31388.27,-49403.25
4,Wheat pair (raw),0.43,0.19,0.34,25.24,-79.71
5,Wheat pair + trend guard,0.48,0.14,0.36,10.60,-40.63
6,Combined (raw),-0.19,1.15,0.16,107757.59,-78607.95
7,Combined + all guards,-0.11,1.13,0.22,98603.83,-64594.44


## What I learned

- **Family taxonomy + IC selection** is a useful diagnostic but doesn't always beat hand-picked   recipes. On this sample, IC selection was best on the PROVIDED data variant. Adding more features   (FULL variant) gave it more rope to overfit IS, not better OOS.
- **Soybeans** are the cleanest commodity. Cargill crush is the strongest single signal.
- **Corn** does not work without an external demand signal. EIA ethanol production/stocks closes   most of the gap. Remaining failure mode is *abundant-supply, low-price* regimes; a simple   below-MA-and-negative-momentum filter goes flat through them.
- **Wheat is not directional** on this sample. The **SRW/HRW pair MR with a 10% Cargill overlay**   is the clean trade. Fractional lot sizing, weekly rebalance, and a signal threshold all matter.
- **Fitted models did not survive** a recipe-vs-OLS comparison by enough to justify the overfit   risk on 10 years of data. I did not test Ridge — regularisation tuning becomes its own research   project on this sample size.

**Next, with more time:**
1. Replace equal-weight family aggregation with feature-level IC weighting inside each family.
2. Add a USDA WASDE event filter — the strategy doesn't currently know about report days.
3. Look at the corn/soybean crush spread as a fourth product.
4. Re-fit recipes on a longer 25-year sample once available.